# Exercise 05: Machine Learning Models
## AIAT 112 — Unit 5

---

## Learning Objectives

In this exercise, you will:
- Build a proper evaluation function with k-fold cross-validation
- Compare three classifier families on the same dataset
- Analyze which model to ship and why

---

## Real-World Context

You are handed the real 1912 Titanic passenger manifest — 891 people, with what a clerk recorded about each of them: travel class, sex, age, family aboard, fare paid, port of embarkation. The task is to predict who survived. Three model families are on the table — logistic regression, decision tree, and k-nearest neighbors. Your job is to evaluate them *fairly* (no testing on training data!) and recommend one.

Two things make this real data rather than a textbook exercise. First, **177 passengers have no recorded age**, so you must decide what to do about that before any model runs. Second, survival was never a clean function of these columns: two passengers with the same ticket and the same age did not always share a fate. That means no model can be perfect here, and the gap between models will be small — which is exactly when careful evaluation earns its keep.

---

## How to work

Each task has a `# YOUR CODE HERE` scaffold. Replace the `raise NotImplementedError` line with your implementation, then run the check cell below it. The check cells run safely even before you implement anything.


## Setup

Run these two cells first — the first sets up the shared data loader (it works from any folder, and on Google Colab), and the second loads the real manifest and turns it into a numeric feature matrix.

In [1]:
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root and put it on sys.path, then import the shared loader.
# WHY:  a hard-coded '../../../Course 04/datasets/raw/titanic.csv' only resolves when the
#       kernel's working directory happens to be this notebook's folder. This does not care.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools.data import load        # load("titanic"), load("crime_statistics_original_50"), ...
# -------------------------------------------------------------------- 8< ----------------
print("Data loader ready. Datasets are found automatically - no file paths in this notebook.")


Data loader ready. Datasets are found automatically - no file paths in this notebook.


In [2]:
# Load the REAL Titanic manifest and turn it into a numeric feature matrix.
# WHY these steps: models need numbers, so categorical columns get encoded and the real
# gaps in 'Age' and 'Embarked' get filled — every one of those is a judgement call you own.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold

titanic = load("titanic")
print(f"Real manifest: {len(titanic)} passengers")
print("Missing values before we do anything:")
print(titanic[["Age", "Embarked", "Fare"]].isna().sum().to_string())

df = titanic.copy()
df["Age"] = df["Age"].fillna(df["Age"].median())          # 177 real gaps -> median age
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])   # 2 real gaps -> commonest port

features = pd.DataFrame({
    "pclass": df["Pclass"],
    "is_female": (df["Sex"] == "female").astype(int),
    "age": df["Age"],
    "family_aboard": df["SibSp"] + df["Parch"],
    "fare": df["Fare"],
    "embarked_C": (df["Embarked"] == "C").astype(int),
    "embarked_Q": (df["Embarked"] == "Q").astype(int),
})

# Standardize so distance-based KNN is not dominated by 'fare' (0-512) over 'pclass' (1-3).
X = ((features - features.mean()) / features.std()).to_numpy(dtype=float)
y = df["Survived"].to_numpy()

print(f"\nDataset: {X.shape[0]} passengers, {X.shape[1]} features")
print(f"Features: {list(features.columns)}")
print(f"Labels: {np.bincount(y)} (died / survived)")
print(f"Baseline to beat — always predict 'died': {(y == 0).mean():.3f}")

titanic: full file, 891 rows.
Real manifest: 891 passengers
Missing values before we do anything:
Age         177
Embarked      2
Fare          0

Dataset: 891 passengers, 7 features
Features: ['pclass', 'is_female', 'age', 'family_aboard', 'fare', 'embarked_C', 'embarked_Q']
Labels: [549 342] (died / survived)
Baseline to beat — always predict 'died': 0.616


## Task 1: Implementation (50 points)

Implement the evaluation function.

**Requirements:**
- `evaluate_model(model, X, y, k=5)` must run **k-fold cross-validation** (use `KFold(n_splits=k, shuffle=True, random_state=0)` with `cross_val_score`, scoring accuracy)
- Return `(mean_accuracy, std_accuracy)` across the folds
- It must NOT fit on the full data and score on the same data — that is the mistake this exercise exists to kill

In [3]:
# TASK 1: implement k-fold cross-validated accuracy (replace `raise NotImplementedError`).
# Why: a single train/test split can be lucky; k folds average away that luck.

def evaluate_model(model, X, y, k=5):
    """k-fold cross-validated accuracy. Returns (mean_accuracy, std_accuracy)."""
    # YOUR CODE HERE
    raise NotImplementedError

print("Function defined — run the check cell below.")

Function defined — run the check cell below.


In [4]:
# --- Check for Task 1 (runs safely before you implement) ---
# The try/except lets this cell run BEFORE you finish Task 1 without crashing the notebook.
# The cross-check against sklearn's own cross_val_score proves your folds match the reference.
try:
    mean_acc, std_acc = evaluate_model(LogisticRegression(max_iter=1000), X, y)
    assert 0.5 < mean_acc < 1.0, "accuracy should beat coin-flipping but not be perfect on this data"
    assert mean_acc > (y == 0).mean(), "a useful model must beat 'always predict died'"
    assert 0.0 <= std_acc < 0.2, "std across folds should be small"
    # cross-check against a direct sklearn call with the same folds
    ref = cross_val_score(LogisticRegression(max_iter=1000), X, y,
                          cv=KFold(n_splits=5, shuffle=True, random_state=0)).mean()
    assert abs(mean_acc - ref) < 1e-9, "must match KFold(5, shuffle=True, random_state=0) exactly"
    print(f"✅ Task 1 check passed: logistic regression CV accuracy = {mean_acc:.3f} ± {std_acc:.3f}")
except NotImplementedError:
    print("⏳ Task 1 not implemented yet — fill in the function above, then re-run this cell.")

⏳ Task 1 not implemented yet — fill in the function above, then re-run this cell.


## Task 2: Evaluation (30 points)

Compare the three candidate models on the Titanic data.

**Requirements:**
- Run the harness below: it evaluates all three models with YOUR `evaluate_model`
- The table must show mean ± std accuracy per model, and name the best one

In [5]:
# --- Evaluation harness (uses your Task 1 implementation) ---
# Why compare three model families with the SAME evaluator? Model choice should be an experiment,
# not a preference - the table makes the decision for you.
try:
    candidates = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree":       DecisionTreeClassifier(random_state=0),
        "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    }
    results = {}
    print(f"{'Model':<22}{'CV accuracy':<15}{'± std'}")
    print("-" * 45)
    for name, model in candidates.items():
        mean_acc, std_acc = evaluate_model(model, X, y)
        results[name] = mean_acc
        print(f"{name:<22}{mean_acc:<15.3f}{std_acc:.3f}")
    best = max(results, key=results.get)
    print()
    print(f"Best cross-validated model on this data: {best} ({results[best]:.3f})")
    print(f"For reference, 'always predict died' scores {(y == 0).mean():.3f}")
except NotImplementedError:
    print("⏳ Complete Task 1 first — this cell reuses evaluate_model.")

Model                 CV accuracy    ± std
---------------------------------------------
⏳ Complete Task 1 first — this cell reuses evaluate_model.


## Task 3: Analysis (20 points)

Answer in the markdown cell below (3-6 sentences each):

1. Report your Task 2 ranking. Are the differences between the models large compared to the ± std across folds? What does that imply about declaring a single "winner"?
2. If you had evaluated by training on all 891 passengers and scoring on the same 891, which of the three models would look misleadingly good, and why?
3. Suppose this model were the prototype for a modern triage tool, where a false negative (predicting "will not survive" for someone who would) costs 5× more than a false alarm. Accuracy ignores that — name a change to the evaluation (metric or threshold) that would respect it.
4. You filled 177 missing ages with the median before cross-validating. Explain why doing that on the *whole* dataset, rather than inside each fold, is a mild form of leakage — and how much you think it matters here.

### Your answers

1. ...

2. ...

3. ...